# IVF Insight — Phase 3: Exploratory Data Analysis & Statistical Investigation

**Author:** Senior Biomedical Data Scientist & Research Engineer  
**Source Dataset:** `data/raw/Dataset Clean.xlsx` (5,000 authoritative patient treatment cycles)  
**Governance Registry:** `reports/feature_governance.csv`  

> [!IMPORTANT]
> **Clinical Non-Claim Notice:** Assisted reproductive technology (ART) involves complex sequential biological decisions. This exploratory analysis is an educational portfolio project. It does **not** establish clinical validity, recommend clinical protocols, or constitute medical advice. Observed statistical associations must **not** be interpreted as causal relationships.

## 1. Setup & Environment

Initialize plotting styles, import statistical packages, and load the `ivf_insight` modular pipeline.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from ivf_insight.eda import (
    load_clean_cohorts,
    compute_descriptive_statistics,
    compute_statistical_associations,
    analyze_data_quality_anomalies,
    run_eda_pipeline,
)

REPORTS_DIR = Path('../reports')
FIGURES_DIR = REPORTS_DIR / 'figures'
print('Imports successful. Reports directory:', REPORTS_DIR.resolve())


## 2. Load Analytical Cohorts

In accordance with Phase 2 feature governance, we define two distinct, non-interchangeable analytical cohorts:
- **Full-Cycle Cohort (N = 5,000):** All initiated cycles, including 1,250 cycles where no embryo transfer occurred (`Non-return of embryos`).
- **Embryo-Transfer Cohort (N = 3,750):** Only cycles with completed intrauterine embryo transfer.

In [ ]:
df_full, df_transfer = load_clean_cohorts('../data/raw/Dataset Clean.xlsx')
print(f'Full-Cycle Cohort: {len(df_full):,} records')
print(f'Embryo-Transfer Cohort: {len(df_transfer):,} records')
print(f'Non-return of embryos count: {(df_full["β-HCG"] == "Non-return of embryos").sum():,}')


## 3. Target Outcome Distribution (`recorded_cycle_outcome`)

Compare the recorded cycle outcome distribution between the two analytical cohorts.

In [ ]:
target_col = 'Output(failed implantation or successful implantation)'

full_target = pd.DataFrame({
    'Count': df_full[target_col].value_counts(),
    'Proportion (%)': (df_full[target_col].value_counts(normalize=True) * 100).round(2)
})
print('=== Full-Cycle Cohort (N=5,000) ===')
display(full_target)

transfer_target = pd.DataFrame({
    'Count': df_transfer[target_col].value_counts(),
    'Proportion (%)': (df_transfer[target_col].value_counts(normalize=True) * 100).round(2)
})
print('=== Embryo-Transfer Cohort (N=3,750) ===')
display(transfer_target)

display(Image(filename=str(FIGURES_DIR / 'target_distribution.png')))


## 4. Patient Demographics: Age and Treatment History

Examine female and male partner age distributions and previous attempt history.

In [ ]:
age_summary = df_full[['Female Age', 'Male Age', 'Number of attempts']].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
display(age_summary)
display(Image(filename=str(FIGURES_DIR / 'age_distributions.png')))


## 5. Anti-Müllerian Hormone (AMH) & Extreme Values

Audit AMH values, right-skewness, and extreme values (including 3 occurrences of 482.1).

In [ ]:
amh = df_full['AMH']
print(f'AMH Range: {amh.min()} to {amh.max()}')
print(f'AMH > 10 count: {(amh > 10).sum()}')
print(f'AMH > 20 count: {(amh > 20).sum()}')
print(f'AMH == 482.1 count: {(amh == 482.1).sum()}')

display(df_full[df_full['AMH'] == 482.1][['Female Age', 'Male Age', 'Egg retrieval date', 'Oocyte Maturity MII', 'Number of fertilized eggs', target_col]])
display(Image(filename=str(FIGURES_DIR / 'amh_distribution.png')))


## 6. Semen Analysis Parameters

Investigate sperm concentration, motility, and morphology distributions.

In [ ]:
sperm_summary = df_full[['Sperm count', 'Sperm motility', 'Sperm morphology']].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
display(sperm_summary)
display(Image(filename=str(FIGURES_DIR / 'sperm_parameters.png')))


## 7. Oocyte Maturity and Fertilization Dynamics

Analyze Metaphase II, Metaphase I, and other oocyte counts relative to fertilization (2PN).

In [ ]:
oocyte_cols = ['Oocyte Maturity MII', 'Oocyte Maturity MI', 'Oocyte Maturity others', 'Number of fertilized eggs']
display(df_full[oocyte_cols].describe().T[['mean', 'std', 'min', '50%', 'max']])
display(Image(filename=str(FIGURES_DIR / 'oocyte_maturity_distribution.png')))


## 8. Embryology Morphological Grades & Vitrification

Inspect embryo grade tiers (1 to 4), total viable embryos, and cryopreservation (`frezzing`).

In [ ]:
embryo_cols = ['Embryo Grade 1', 'Embryo Grade 2', 'Embryo Grade 3', 'Embryo Grade 4', 'Number of embryos', 'frezzing']
display(df_full[embryo_cols].describe().T[['mean', 'std', 'min', '50%', 'max']])
display(Image(filename=str(FIGURES_DIR / 'embryo_grades_distribution.png')))


## 9. Temporal Outcome Trends (2019–2025)

Evaluate annual outcome prevalence across the 7-year retrieval timeline.

In [ ]:
df_full['Year'] = pd.to_datetime(df_full['Egg retrieval date']).dt.year
df_transfer['Year'] = pd.to_datetime(df_transfer['Egg retrieval date']).dt.year

yearly_table = pd.DataFrame({
    'Full_N': df_full.groupby('Year')[target_col].count(),
    'Full_Pos_Rate (%)': (df_full.groupby('Year')[target_col].mean() * 100).round(2),
    'Transfer_N': df_transfer.groupby('Year')[target_col].count(),
    'Transfer_Pos_Rate (%)': (df_transfer.groupby('Year')[target_col].mean() * 100).round(2),
})
display(yearly_table)
display(Image(filename=str(FIGURES_DIR / 'temporal_outcome_trend.png')))


## 10. Feature Correlation Matrix (Spearman Rank Correlation)

Examine multi-collinearity and rank relationships among treatment parameters.

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'correlation_heatmap.png')))


## 11. Statistical Hypothesis Testing with Benjamini–Hochberg FDR Control

Review univariate hypothesis tests (Mann–Whitney U for continuous features, Chi-square for categorical features) evaluated separately on both cohorts.

In [ ]:
stats_df = pd.read_csv('../reports/statistical_tests.csv')

print('=== Full-Cycle Cohort: Top Significant Features (BH-FDR q < 0.05) ===')
display(stats_df[stats_df['cohort'].str.contains('Full-Cycle')].sort_values('q_value_bh')[['feature', 'test_type', 'statistic', 'p_value', 'q_value_bh', 'effect_size_metric', 'effect_size']])

print('=== Embryo-Transfer Cohort: Top Significant Features (BH-FDR q < 0.05) ===')
display(stats_df[stats_df['cohort'].str.contains('Embryo-Transfer')].sort_values('q_value_bh')[['feature', 'test_type', 'statistic', 'p_value', 'q_value_bh', 'effect_size_metric', 'effect_size']])

display(Image(filename=str(FIGURES_DIR / 'cohort_comparison_transfer_vs_nonreturn.png')))


## 12. Summary of Findings & Non-Claim Notice

1. **Cohort Sensitivity:** Positive outcome rate is 49.84% in the full cohort but 66.45% in the embryo-transfer cohort. Effect sizes for embryonic variables are vastly inflated if non-transferred cycles are conflated with transfer cycles.
2. **Age Significance:** Female Age is statistically significant in the transfer cohort ($q = 0.00049$) but masked in the full cohort ($q = 0.283$).
3. **Top Prognostic Signals (Transfer Cohort):** Surplus cryopreserved embryos (`frezzing`, $r = 0.355$), fertilized eggs ($r = 0.313$), top-grade embryos ($r = 0.275$), mature MII oocytes ($r = 0.274$), and protocol ID (Cramer's $V = 0.336$).
4. **Data Quality Anomalies:** Three occurrences of AMH = 482.1, 182 protocol IDs with heavy-tailed distribution (ID 1 = 16.3%), and 549 cycles where fertilized eggs exceed MII oocytes (reflecting IVM or non-denuded IVF).

**No machine learning models were trained. No clinical validity is claimed.**